# Six Sigma DMAIC — Improve & Control Phase

Simulates the effect of the improvement action on the Network category
and compares process capability before vs. after.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent
RAW  = PROJECT_ROOT / "data" / "raw"
PROC = PROJECT_ROOT / "data" / "processed"
CHARTS = PROJECT_ROOT / "docs" / "screenshots"
PROC.mkdir(parents=True, exist_ok=True)

SLA_HOURS = 24

df = pd.read_csv(RAW / "helpdesk_tickets.csv", parse_dates=["created_at"])
network_before = df[df["category"] == "Network"]["resolution_hours"].values

In [ ]:
def imr_limits(x):
    mr = np.abs(np.diff(x))
    sigma = mr.mean() / 1.128
    x_bar = x.mean()
    ucl = x_bar + 3 * sigma
    lcl = max(0, x_bar - 3 * sigma)
    return x_bar, sigma, ucl, lcl

def cpk_upper(x, usl):
    x_bar, sigma, _, _ = imr_limits(x)
    return (usl - x_bar) / (3 * sigma)

### Baseline capability (before improvement)

Improvement actions defined in `dmaic/04_improve.md`: standardized
triage checklist, first-line diagnostics documentation, and a defined
escalation SLA with the ISP provider.

In [ ]:
x_bar_before, sigma_before, ucl_before, lcl_before = imr_limits(network_before)
cpk_before = cpk_upper(network_before, SLA_HOURS)

print(f"Mean MTTR (before)     : {x_bar_before:.1f}h")
print(f"Sigma (before)          : {sigma_before:.1f}")
print(f"Cpk (before)             : {cpk_before:.2f}")
print(f"SLA compliance (before)  : {(network_before <= SLA_HOURS).mean()*100:.1f}%")

### Simulated improvement

The improvement target mirrors a real result achieved in a comparable
DMAIC cycle (technical support ticket workflow, -13% MTTR). Applied
here as a -13% shift in mean resolution time plus a proportional
reduction in variability -- the expected effect of standardizing the
triage process.

In [ ]:
np.random.seed(99)
improvement_factor = 0.87  # -13%
network_after = network_before * improvement_factor * np.random.normal(1.0, 0.05, len(network_before))
network_after = np.clip(network_after, 0.5, None)

x_bar_after, sigma_after, ucl_after, lcl_after = imr_limits(network_after)
cpk_after = cpk_upper(network_after, SLA_HOURS)

print(f"Mean MTTR (after)      : {x_bar_after:.1f}h")
print(f"Sigma (after)           : {sigma_after:.1f}")
print(f"Cpk (after)              : {cpk_after:.2f}")
print(f"SLA compliance (after)   : {(network_after <= SLA_HOURS).mean()*100:.1f}%")
print()
print(f"MTTR reduction: {(1 - x_bar_after/x_bar_before)*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

chart_data = [
    (axes[0], network_before, x_bar_before, ucl_before, lcl_before, "Before"),
    (axes[1], network_after,  x_bar_after,  ucl_after,  lcl_after,  "After"),
]
for ax, data, xbar, ucl_v, lcl_v, title in chart_data:
    ax.plot(data, marker="o", markersize=2, linewidth=0.7, color="#4C72B0")
    ax.axhline(xbar, color="black", linewidth=1)
    ax.axhline(ucl_v, color="#C44E52", linestyle="--", linewidth=1)
    ax.axhline(lcl_v, color="#C44E52", linestyle="--", linewidth=1)
    ax.axhline(SLA_HOURS, color="#55A868", linestyle=":", linewidth=1.5)
    ax.set_title(f"Network -- {title}", fontsize=12)
    ax.set_xlabel("Ticket sequence")

axes[0].set_ylabel("Resolution time (hours)")
plt.tight_layout()
plt.savefig(CHARTS / "control_chart_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

### Control plan and KPI export

Ongoing monitoring defined in `dmaic/05_control.md`: weekly control
chart review, monthly Cpk recalculation, and an alert rule if 2+
consecutive points exceed the UCL.

In [ ]:
summary = pd.DataFrame({
    "metric": ["mean_mttr_hours", "sigma", "cpk", "sla_compliance_pct"],
    "before": [round(x_bar_before, 2), round(sigma_before, 2), round(cpk_before, 2),
               round((network_before <= SLA_HOURS).mean() * 100, 1)],
    "after":  [round(x_bar_after, 2), round(sigma_after, 2), round(cpk_after, 2),
               round((network_after <= SLA_HOURS).mean() * 100, 1)],
})
summary.to_csv(PROC / "dmaic_kpi_summary.csv", index=False)
print(summary.to_string(index=False))